<a href="https://colab.research.google.com/github/ksehar99/DR-blockchain/blob/main/DR_Blockchain_Web3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# # Kaggle credentials
# import os
os.environ['KAGGLE_USERNAME'] = 'khadija4058'
os.environ['KAGGLE_KEY'] = 'KGAT_61e373e830995974eee3022ca7f6673f'

# # Download
# !kaggle datasets download -d mariaherrerot/aptos2019
# !unzip aptos2019.zip -d /content/drive/MyDrive/aptos2019

Dataset URL: https://www.kaggle.com/datasets/mariaherrerot/aptos2019
License(s): unknown
  7% 579M/8.01G [00:05<01:08, 117MB/s]
User cancelled operation
Archive:  aptos2019.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of aptos2019.zip or
        aptos2019.zip.zip, and cannot find aptos2019.zip.ZIP, period.


In [4]:
!pip install web3
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.0/344.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.1 MB/s eta 0:00:00


# Connecting with Sepolia Testnet

In [5]:
from web3 import Web3
from google.colab import userdata

# Secrets se load karo
ALCHEMY_URL = userdata.get('ALCHEMY_URL')
PRIVATE_KEY = userdata.get('MMprivatekey')

# Connect to Sepolia
w3 = Web3(Web3.HTTPProvider(ALCHEMY_URL))
print("Connected:", w3.is_connected())

# Wallet address
account = w3.eth.account.from_key(PRIVATE_KEY)
owner_address = account.address
print("Owner Address:", owner_address)

# Contract setup
CONTRACT_ADDRESS = "0x922EcbF05cB5f404386f2ca4bAfDD2ac6BEbff8a"
ABI = [{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[],"name":"NotAuthorized","type":"error"},{"inputs":[],"name":"NotOwner","type":"error"},{"inputs":[],"name":"PatientAlreadyExist","type":"error"},{"anonymous":False,"inputs":[{"indexed":False,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":False,"internalType":"uint256","name":"diagnosisResut","type":"uint256"},{"indexed":False,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"DiagnosisUploaded","type":"event"},{"anonymous":False,"inputs":[{"indexed":False,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":False,"internalType":"address","name":"doctorAddress","type":"address"}],"name":"PatientRegistered","type":"event"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"},{"internalType":"address","name":"_doctorAddress","type":"address"}],"name":"registerPatient","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"bytes32","name":"imageHash","type":"bytes32"},{"internalType":"uint256","name":"diagnosisResult","type":"uint256"}],"name":"uploadDiagnosis","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"viewPatients","outputs":[{"internalType":"uint256[]","name":"","type":"uint256[]"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"}],"name":"viewRecords","outputs":[{"components":[{"internalType":"bytes32","name":"diagnosisImageHash","type":"bytes32"},{"internalType":"uint256","name":"diagnosisResultCategory","type":"uint256"},{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"address","name":"doctorId","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"}],"internalType":"struct DRDiagnosisResuls.Diagnosis[]","name":"","type":"tuple[]"}],"stateMutability":"view","type":"function"}]
contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=ABI)
print("Contract loaded!")

Connected: True
Owner Address: 0x8830342a7937361ac4F4407255c8a98ea9A1D775
Contract loaded!


# Register Patient

In [11]:
from os import times
from faker import Faker
import json
import random
import datetime
import os

def create_Fake_Patient():
  # load data from json
  JSON_PATH = '/content/drive/MyDrive/patients.json'

  if not os.path.exists(JSON_PATH):
    with open (JSON_PATH, 'w') as f:
      json.dump([], f, indent = 4)
    data = []
  else:
    with open (JSON_PATH, 'r') as f:
      data = json.load(f)

  # create fake patient
  pat_new_id = len(data)+1
  # doctor1 = "0x5B38Da6a701c568545dCfcB03FcB875f56beddC4"
  # doctor2 = "0xAb8483F64d9C6d1EcF9b849Ae677dD3315835cb2"
  doctor1 = owner_address
  doctor2 = owner_address

  faker = Faker()
  DOCTORS = {
    "doctor1": owner_address,
    "doctor2": owner_address  # abhi dono same
}
  doctorName = random.choice(["doctor1", 'doctor2'])
  patient = {
      "patientId" : pat_new_id,
      "doctorName": doctorName,
      "doctorAddress" : DOCTORS[doctorName],
      "name" : faker.name(),
      "timestamp" : datetime.datetime.now().isoformat()
  }

  print(f"Patient created: {patient}")
  return patient, data


In [7]:
def register_patient(patient, data):
  #  make transaction
  txn = contract.functions.registerPatient(patient["patientId"], patient["doctorAddress"]).build_transaction({
    'from': owner_address,
    'nonce': w3.eth.get_transaction_count(owner_address, "pending"), # nonce is a txn no. increment on each txn.. and prevent from reply attack "hr nonce sirf ek hi baar chlta"
    'gas': 200000,
    'gasPrice': w3.eth.gas_price
  })
  signed_txn = w3.eth.account.sign_transaction(txn, PRIVATE_KEY)
  # when sign trnsaction it includes 2 things: raw transaction, and other is hash of the transaction
  txn_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction) # sendRAwTransaction - sends raw transaction to the blockchain and retun the hash of transaction

  # txn_hash and signed_txn.hash have same values, difference is that txn_hash shows confirmed transaction-means txn is deployed and verified by blockchain

  # wait for the transaction to aprove
  receipt = w3.eth.wait_for_transaction_receipt(txn_hash)

  print(f'transaction succeded: {bool(receipt.status)}')
  print(f'transction hash: {txn_hash.hex()}')

  JSON_PATH = '/content/drive/MyDrive/patients.json'

  if receipt.status == 1:
    # write data to the json
    data.append(patient)

    with open(JSON_PATH, 'w') as f:
      json.dump(data, f, indent = 4)

    print(f"Patient added to JSON: {patient}")

  return receipt.status == 1

# w3.eth.get_transaction_count()  → nonce lo
# w3.eth.gas_price                → current gas price
# w3.eth.send_raw_transaction()   → transaction bhejo
# w3.eth.wait_for_transaction_receipt() → confirm hone ka wait karo
# w3.eth.account                  → wallet related functions

In [14]:
# # uncomment and run this if redploy the contract
# import os
# JSON_PATH = '/content/drive/MyDrive/patients.json'

# if os.path.exists(JSON_PATH):
#     os.remove(JSON_PATH)
# print("Done!")

Done!


In [37]:
patient, data = create_Fake_Patient()
register_patient(patient, data)

Patient created: {'patientId': 5, 'doctorName': 'doctor2', 'doctorAddress': '0x8830342a7937361ac4F4407255c8a98ea9A1D775', 'name': 'Courtney Ramirez', 'timestamp': '2026-06-25T11:01:58.965455'}
transaction succeded: True
transction hash: d0c7deeef0d6fc7119528c6545f89eb344d12431b67fed63058e7685f9103e3f
Patient added to JSON: {'patientId': 5, 'doctorName': 'doctor2', 'doctorAddress': '0x8830342a7937361ac4F4407255c8a98ea9A1D775', 'name': 'Courtney Ramirez', 'timestamp': '2026-06-25T11:01:58.965455'}


True

# Model Prediction

In [22]:
import pandas as pd
import os
import tensorflow as tf

test_path = '/content/drive/MyDrive/aptos2019/test_images/test_images'
test_images = os.listdir(test_path)
test_csv = pd.read_csv('/content/drive/MyDrive/aptos2019/test.csv')

print(f"Total test images: {len(test_images)} \n")

model = tf.keras.models.load_model('/content/drive/MyDrive/dr_model_phase2.h5')
print("Model loaded!")

Total test images: 366 



Model loaded!


In [30]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input

used_images = []
def predict_diagnosis(test_images, test_csv, model):
    # Selct Random Image
    available = [img for img in test_images if img not in used_images]
    if len(available) == 0:
        return None, None, None
    image_filename = random.choice(available)
    image_path = f'/content/drive/MyDrive/aptos2019/test_images/test_images/{image_filename}'
    print(f"Selected image: {image_filename}")

    # Check its true label
    class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']

    image_name = image_filename.replace('.png', '')
    real_label = test_csv[test_csv['id_code'] == image_name]['diagnosis'].values[0]
    print(f"Real label: {class_names[real_label]}")

    # Model prediction
    img = load_img(image_path, target_size=(224, 224))
    img_array = preprocess_input(np.expand_dims(img_to_array(img), axis=0))
    predicted_class = int(np.argmax(model.predict(img_array)))
    print(f"Predicted: {class_names[predicted_class]} (Class {predicted_class})")

    return image_filename, image_path, predicted_class

In [35]:
import hashlib

def upload_diagnosis(patient_id, image_filename, predicted_class, data):
  image_path = f'/content/drive/MyDrive/aptos2019/test_images/test_images/{image_filename}'

  patient = next((p for p in data if p["patientId"] == patient_id), None)
  if patient is None:
      print("Patient not found!")
      return False

  # Calculate the hash of the image
  with open(image_path, 'rb') as f:
      image_bytes = f.read()

  image_hash = hashlib.sha256(image_bytes).hexdigest()
  image_hash_bytes32 = bytes.fromhex(image_hash)

  # make transsactions
  print(f"DEBUG: Building transaction with 'from' address: {owner_address}")
  txn = contract.functions.uploadDiagnosis(patient["patientId"], image_hash_bytes32, predicted_class).build_transaction({
    'from': owner_address,
    'nonce': w3.eth.get_transaction_count(owner_address, "pending"), # nonce is a txn no. increment on each txn.. and prevent from reply attack "hr nonce sirf ek hi baar chlta"
    'gas': 200000,
    'gasPrice': w3.eth.gas_price
  })
  signed_txn = w3.eth.account.sign_transaction(txn, PRIVATE_KEY)
  # when sign trnsaction it includes 2 things: raw transaction, and other is hash of the transaction
  txn_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction) # sendRAwTransaction - sends raw transaction to the blockchain and retun the hash of transaction

  # txn_hash and signed_txn.hash have same values, difference is that txn_hash shows confirmed transaction-means txn is deployed and verified by blockchain

  # wait for the transaction to aprove
  receipt = w3.eth.wait_for_transaction_receipt(txn_hash)

  print(f'transaction succeded: {bool(receipt.status)}')
  print(f'transction hash: {txn_hash.hex()}')

  JSON_PATH = '/content/drive/MyDrive/patients.json'

  if receipt.status == 1:
    used_images.append(image_filename)
    # JSON mein patient update karo
    for p in data:
        if p["patientId"] == patient["patientId"]:
            p["image_filename"] = image_filename
            p["predicted_class"] = predicted_class
            break
    with open(JSON_PATH, 'w') as f:
        json.dump(data, f, indent=4)
    print(f"Diagnosis uploaded for patient {patient['patientId']}")

  return receipt.status == 1

In [38]:

# 2. Image select karo aur predict karo
image_filename, image_path, predicted_class = predict_diagnosis(test_images, test_csv, model)

# 3. Diagnosis upload karo
upload_diagnosis(4, image_filename, predicted_class, data)

Selected image: ebd96d853918.png
Real label: No DR
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step
Predicted: Proliferative DR (Class 4)
DEBUG: Building transaction with 'from' address: 0x8830342a7937361ac4F4407255c8a98ea9A1D775
transaction succeded: True
transction hash: b3066286b013ce067c758c7a2ef1b5a7908556628262db35555e10952bdd0dfb
Diagnosis uploaded for patient 4


True